In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
# ---- Config ----
CATALOG = "workspace"

SILVER_TABLE = f"{CATALOG}.careerpulse_silver.job_postings"

COUNTS_TABLE = f"{CATALOG}.careerpulse_gold.job_title_daily_counts"
FEATURES_TABLE = f"{CATALOG}.careerpulse_gold.job_demand_features_daily"
CATEGORY_TABLE = f"{CATALOG}.careerpulse_gold.category_labeled_postings"

In [0]:
from pyspark.sql.functions import lit, col, to_date, lower, count
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver = spark.table(SILVER_TABLE)

# Aggregate daily posting counts by category directly from Silver
counts = (
    silver
    .filter(col("category").isNotNull())
    .groupBy(
        col("category"),
        col("category_short"),
        to_date(col("posted_at")).alias("date")
    )
    .agg(F.count("*").alias("n_postings"))
)

counts.createOrReplaceTempView("counts")

# Upsert into gold counts table
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {COUNTS_TABLE}
    USING DELTA
    AS SELECT * FROM counts
    WHERE 1 = 0
""")

spark.sql(f"""
    MERGE INTO {COUNTS_TABLE} AS t
    USING counts AS c
    ON t.category = c.category
    AND t.date = c.date
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
#                                                               ---- Update gold.job_demands_feature_daily table ----

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load daily counts
counts = (
    spark.table(COUNTS_TABLE)
    .select(
        col("category"), 
        col("category_short"), 
        F.col("date").cast("date").alias("date"),
        F.col("n_postings").cast("int").alias("y")
        )
)

# Make the time series continuous per query (fill missing dates with y=0)
date_bounds = counts.groupBy("category", "category_short").agg(
    F.min(F.col("date")).alias("min_date"),
    F.max(F.col("date")).alias("max_date")
)

date_spine = (
    date_bounds
    .withColumn("date", F.explode(F.sequence(F.col("min_date"), F.col("max_date"))))
    .select("category", "category_short", "date")
)

ts = (
    date_spine
    .join(counts, on=["category", "category_short", "date"], how="left")
    .fillna({"y": 0})
)

# Feature engineering
w = Window.partitionBy("category").orderBy("date")
w7 = w.rowsBetween(-6, 0)
w14 = w.rowsBetween(-13, 0)
w30 = w.rowsBetween(-29, 0)

features = (
    ts
    # calendar features
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("is_weekend", F.col("day_of_week").isin([1, 7]).cast("int"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day_of_month", F.dayofmonth("date"))

    # lags
    .withColumn("y_lag_1",  F.lag("y", 1).over(w))
    .withColumn("y_lag_7",  F.lag("y", 7).over(w))
    .withColumn("y_lag_14", F.lag("y", 14).over(w))

    # rolling stats (using past values up to today)
    .withColumn("rolling_mean_7",  F.avg("y").over(w7))
    .withColumn("rolling_std_7",   F.stddev("y").over(w7))
    .withColumn("rolling_mean_14", F.avg("y").over(w14))
    .withColumn("rolling_mean_30", F.avg("y").over(w30))

    # simple change
    .withColumn("y_change_1", F.col("y") - F.lag("y", 1).over(w))
    .withColumn("y_change_7", F.col("y") - F.lag("y", 7).over(w))
)

features.createOrReplaceTempView("features")

# create the features table (if not exists)
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {FEATURES_TABLE}
          USING DELTA
          AS
          SELECT * FROM features
          WHERE 1 = 0
          """)

# merge new rows into the features table
spark.sql(f"""
            MERGE INTO {FEATURES_TABLE} AS t
            USING features AS f
            ON t.category = f.category
            AND t.date = f.date
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
            """)
#

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from utils.clean_description import clean_description

clean_description_udf = udf(lambda x: clean_description(x), StringType())

cat_labeled = (
    silver
    .select("posting_id", "category", "title", "description")
    .filter(F.col("category").isNotNull())
    .withColumn("description_clean", clean_description_udf(F.col("description")))
    .filter(F.col("description_clean").isNotNull())
    .filter(F.col("description_clean") != "")
    .drop("description")
    .dropDuplicates(["posting_id"])
    )

cat_labeled.createOrReplaceTempView("category_labeled")

# create the cat_labeled table (if not exists) using incoming table schema
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATEGORY_TABLE}
          USING DELTA
          AS
          SELECT * FROM category_labeled
          WHERE 1 = 0
          """)

# merge new rows into the features table
spark.sql(f"""
            MERGE INTO {CATEGORY_TABLE} AS t
            USING category_labeled AS c
            ON t.posting_id = c.posting_id
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
            """)


In [0]:
# dbutils.library.restartPython()